# Antibiotic Policy Engine for Indian ICUs

**What this does:** picks an empiric antibiotic for a clinical syndrome by combining three things that usually live in separate places:

1. **National guideline**: ICMR *Treatment Guidelines for Antimicrobial Use in Common Syndromes* (which drugs are allowed for this syndrome)
2. **This ICU's own resistance data**: a WHONET isolate export, turned into a cumulative antibiogram (which of those drugs still work here)
3. **This hospital's stewardship policy**: AWaRe class, restricted drugs, formulary, patient contraindications (which of the working drugs we're allowed to use first)

**Design choice:** the recommendation itself is **rule-based and deterministic**, not an LLM. Antibiotic advice has to be reproducible and traceable to a number. An LLM is used only in section 7, to *explain* stewardship principles from the ICMR text (RAG).

**Run:** `Runtime → Run all`. It takes about 2 minutes without the RAG section and about 8 minutes with it (the Ollama model download).

⚠️ Educational prototype. The antibiogram data are synthetic. Not for clinical use.

## 1. Setup

In [ ]:
!pip -q install pypdf faiss-cpu langchain langchain-community langchain-ollama gradio 2>/dev/null | tail -1
import os, json, time, zipfile, urllib.request
import pandas as pd, numpy as np, matplotlib.pyplot as plt
os.makedirs('outputs', exist_ok=True)
print('ready')

## 2. Build the input data
Writes `make_data.py` and runs it. It produces the ICMR regimen table, the syndrome→specimen map, the hospital policy, and synthetic WHONET exports for **two ICUs**: A (tertiary, high resistance) and B (secondary, lower resistance). To use a real ICU, replace `data/icu_*_whonet.csv` with your own export in the same columns.

In [ ]:
%%writefile make_data.py
"""
Builds the input files for the Antibiotic Policy Engine.

- data/icmr_regimens.csv      syndrome-wise empiric regimens curated from the ICMR
                              Treatment Guidelines for Antimicrobial Use in Common Syndromes (2019)
- data/specimen_map.csv       syndrome -> clinical specimen type
- data/policy.csv             hospital stewardship policy (AWaRe class, restriction, formulary, contraindications)
- data/icu_A_whonet.csv,      synthetic WHONET-style isolate exports for two ICUs
  data/icu_B_whonet.csv       (A = tertiary, high resistance; B = secondary, lower resistance)

The resistance rates are SYNTHETIC. They are set to roughly the ranges published in
ICMR AMRSN annual reports for Indian ICUs, so the engine behaves realistically, but
they are not any real hospital's data. Replace the two WHONET files with a real export
to run the engine for a real ICU.
"""
import os
import numpy as np, pandas as pd

OUT = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'data')
os.makedirs(OUT, exist_ok=True)

# ------------------------------------------------------------------ ICMR regimens
REG = [
 # syndrome, line, regimen, drugs, dosing (adult, normal renal function), source
 ('Acute undifferentiated fever (severe)', 1, 'Ceftriaxone + Doxycycline', 'CRO+DOX', 'Ceftriaxone 2 g IV q24h + Doxycycline 100 mg IV/PO q12h', 'ICMR 2019 Ch.2 (enteric fever + scrub typhus cover)'),
 ('Acute undifferentiated fever (severe)', 2, 'Piperacillin-tazobactam + Doxycycline', 'TZP+DOX', 'Pip-tazo 4.5 g IV q6h + Doxycycline 100 mg q12h', 'ICMR 2019 Table 2.1'),
 ('Acute undifferentiated fever (severe)', 3, 'Meropenem + Doxycycline', 'MEM+DOX', 'Meropenem 1 g IV q8h + Doxycycline 100 mg q12h', 'ICMR 2019 Table 2.1 (seriously ill)'),
 ('Sepsis / septic shock (unclear focus)', 1, 'Imipenem + Vancomycin', 'IPM+VAN', 'Imipenem 1 g IV q8h + Vancomycin 15 mg/kg IV q8-12h', 'ICMR 2019 Table 3.1 (preferred)'),
 ('Sepsis / septic shock (unclear focus)', 2, 'Meropenem + Vancomycin', 'MEM+VAN', 'Meropenem 1 g IV q8h + Vancomycin 15 mg/kg IV q8-12h', 'ICMR 2019 Table 3.1 (alternative)'),
 ('Sepsis / septic shock (unclear focus)', 2, 'Cefoperazone-sulbactam + Amikacin + Teicoplanin', 'CSL+AMK+TEC', 'Cefop-sulb 3 g IV q12h + Amikacin 15 mg/kg q24h + Teicoplanin 400 mg q12h x3 then daily', 'ICMR 2019 Table 3.1 (alternative)'),
 ('Sepsis / septic shock (unclear focus)', 3, 'Meropenem + Colistin + Vancomycin', 'MEM+COL+VAN', 'Meropenem 1 g q8h + Colistin 9 MU load then 4.5 MU q12h + Vancomycin', 'ICMR 2019 Table 3.1 (CRE risk)'),
 ('Sepsis / septic shock (unclear focus)', 3, 'Meropenem + Polymyxin B + Teicoplanin', 'MEM+PMB+TEC', 'Meropenem 1 g q8h + Polymyxin B 15-20 lakh U load then 7.5-10 lakh U q12h + Teicoplanin', 'ICMR 2019 Table 3.1 (CRE risk)'),
 ('Severe community-acquired pneumonia (ICU)', 1, 'Ceftriaxone + Azithromycin', 'CRO+AZM', 'Ceftriaxone 2 g IV q24h + Azithromycin 500 mg IV q24h', 'ICMR 2019 Ch.4 (CAP)'),
 ('Severe community-acquired pneumonia (ICU)', 2, 'Amoxicillin-clavulanate + Azithromycin', 'AMC+AZM', 'Amox-clav 1.2 g IV q8h + Azithromycin 500 mg q24h', 'ICMR 2019 Ch.4 (CAP)'),
 ('Severe community-acquired pneumonia (ICU)', 3, 'Piperacillin-tazobactam + Azithromycin', 'TZP+AZM', 'Pip-tazo 4.5 g IV q6h + Azithromycin 500 mg q24h', 'ICMR 2019 Ch.4 (Pseudomonas risk)'),
 ('Hospital-acquired / ventilator-associated pneumonia', 1, 'Piperacillin-tazobactam', 'TZP', 'Pip-tazo 4.5 g IV q6h (extended infusion)', 'ICMR 2019 (no MDR risk)'),
 ('Hospital-acquired / ventilator-associated pneumonia', 1, 'Cefoperazone-sulbactam', 'CSL', 'Cefop-sulb 3 g IV q12h', 'ICMR 2019 (no MDR risk)'),
 ('Hospital-acquired / ventilator-associated pneumonia', 2, 'Meropenem', 'MEM', 'Meropenem 1 g IV q8h (extended infusion)', 'ICMR 2019 (MDR risk)'),
 ('Hospital-acquired / ventilator-associated pneumonia', 3, 'Meropenem + Colistin', 'MEM+COL', 'Meropenem 1 g q8h + Colistin 9 MU load then 4.5 MU q12h', 'ICMR 2019 (CR-GNB risk)'),
 ('Hospital-acquired / ventilator-associated pneumonia', 3, 'Meropenem + Colistin + Linezolid', 'MEM+COL+LNZ', 'as above + Linezolid 600 mg IV q12h', 'ICMR 2019 (CR-GNB + MRSA risk)'),
 ('Intra-abdominal infection (secondary peritonitis)', 1, 'Piperacillin-tazobactam', 'TZP', 'Pip-tazo 4.5 g IV q6h', 'ICMR 2019 Sec.2.1.6 (less severe)'),
 ('Intra-abdominal infection (secondary peritonitis)', 1, 'Cefoperazone-sulbactam', 'CSL', 'Cefop-sulb 3 g IV q12h', 'ICMR 2019 Sec.2.1.6 (less severe)'),
 ('Intra-abdominal infection (secondary peritonitis)', 2, 'Ertapenem', 'ETP', 'Ertapenem 1 g IV q24h', 'ICMR 2019 Sec.2.1.6 (severe)'),
 ('Intra-abdominal infection (secondary peritonitis)', 2, 'Meropenem', 'MEM', 'Meropenem 1 g IV q8h', 'ICMR 2019 Sec.2.1.6 (severe)'),
]
pd.DataFrame(REG, columns=['SYNDROME', 'LINE', 'REGIMEN', 'DRUGS', 'DOSING', 'SOURCE']).to_csv(f'{OUT}/icmr_regimens.csv', index=False)

SPEC = {'Acute undifferentiated fever (severe)': 'Blood', 'Sepsis / septic shock (unclear focus)': 'Blood',
        'Severe community-acquired pneumonia (ICU)': 'Resp-CAP', 'Hospital-acquired / ventilator-associated pneumonia': 'Resp-VAP',
        'Intra-abdominal infection (secondary peritonitis)': 'Peritoneal'}
pd.DataFrame(list(SPEC.items()), columns=['SYNDROME', 'SPEC_TYPE']).to_csv(f'{OUT}/specimen_map.csv', index=False)

# ------------------------------------------------------------------ stewardship policy
POL = [
 # code, name, AWaRe, restriction, formulary, culture-evaluable, contraindicated in
 ('AMC', 'Amoxicillin-clavulanate', 'Access', 'free', 1, 1, 'penicillin_anaphylaxis'),
 ('AMK', 'Amikacin', 'Access', 'free', 1, 1, 'aki'),
 ('DOX', 'Doxycycline', 'Access', 'free', 1, 0, 'none'),
 ('CRO', 'Ceftriaxone', 'Watch', 'free', 1, 1, 'none'),
 ('AZM', 'Azithromycin', 'Watch', 'free', 1, 0, 'none'),
 ('TZP', 'Piperacillin-tazobactam', 'Watch', 'justify', 1, 1, 'penicillin_anaphylaxis'),
 ('CSL', 'Cefoperazone-sulbactam', 'Watch', 'justify', 1, 1, 'none'),
 ('ETP', 'Ertapenem', 'Watch', 'justify', 1, 1, 'none'),
 ('IPM', 'Imipenem', 'Watch', 'id_approval', 1, 1, 'none'),
 ('MEM', 'Meropenem', 'Watch', 'id_approval', 1, 1, 'none'),
 ('VAN', 'Vancomycin', 'Watch', 'justify', 1, 1, 'none'),
 ('TEC', 'Teicoplanin', 'Watch', 'justify', 1, 1, 'none'),
 ('COL', 'Colistin', 'Reserve', 'id_approval', 1, 1, 'aki'),
 ('PMB', 'Polymyxin B', 'Reserve', 'id_approval', 1, 1, 'aki'),
 ('LNZ', 'Linezolid', 'Reserve', 'id_approval', 1, 1, 'none'),
]
pd.DataFrame(POL, columns=['CODE', 'NAME', 'AWARE', 'RESTRICTION', 'FORMULARY', 'CULTURE_EVALUABLE', 'CONTRAINDICATED_IN']).to_csv(f'{OUT}/policy.csv', index=False)

# ------------------------------------------------------------------ synthetic WHONET isolates
DRUGS = ['AMC', 'CRO', 'TZP', 'CSL', 'ETP', 'IPM', 'MEM', 'AMK', 'COL', 'PMB', 'VAN', 'TEC', 'LNZ', 'OXA', 'AMP']
GNB = {'Escherichia coli', 'Klebsiella pneumoniae', 'Acinetobacter baumannii', 'Pseudomonas aeruginosa', 'Salmonella Typhi', 'Haemophilus influenzae'}

def phenotype(org, rng, k):
    """Draw one isolate's S/R results with realistic co-resistance. k scales resistance (ICU-B < ICU-A)."""
    r = {d: None for d in DRUGS}
    R = lambda p: 'R' if rng.rand() < min(1, p * k) else 'S'
    if org in ('Escherichia coli', 'Klebsiella pneumoniae'):
        cr_p, esbl_p, amk_p, col_p = (0.25, 0.60, 0.15, 0.02) if org == 'Escherichia coli' else (0.55, 0.30, 0.25, 0.06)
        cr = rng.rand() < cr_p * k; esbl = cr or rng.rand() < esbl_p * min(1, k * 1.2)
        for d in ('IPM', 'MEM', 'ETP'): r[d] = 'R' if cr else 'S'
        r['CRO'] = 'R' if esbl else 'S'; r['AMC'] = 'R' if esbl else R(0.15)
        r['TZP'] = 'R' if cr else (R(0.40) if esbl else R(0.05)); r['CSL'] = 'R' if cr else (R(0.35) if esbl else R(0.05))
        r['AMK'] = R(amk_p + (0.5 if cr else 0)); r['COL'] = R(col_p); r['PMB'] = r['COL']; r['AMP'] = 'R'
    elif org == 'Acinetobacter baumannii':
        cr = rng.rand() < 0.85 * k
        for d in ('IPM', 'MEM'): r[d] = 'R' if cr else 'S'
        for d in ('AMC', 'CRO', 'ETP', 'AMP'): r[d] = 'R'          # intrinsic / unreliable
        r['TZP'] = 'R' if cr else R(0.3); r['CSL'] = R(0.75) if cr else 'S'
        r['AMK'] = R(0.70); r['COL'] = R(0.03); r['PMB'] = r['COL']
    elif org == 'Pseudomonas aeruginosa':
        cr = rng.rand() < 0.35 * k
        for d in ('IPM', 'MEM'): r[d] = 'R' if cr else 'S'
        for d in ('AMC', 'CRO', 'ETP', 'AMP'): r[d] = 'R'          # intrinsic
        r['TZP'] = R(0.8) if cr else R(0.25); r['CSL'] = R(0.7) if cr else R(0.25)
        r['AMK'] = R(0.2 + (0.4 if cr else 0)); r['COL'] = R(0.03); r['PMB'] = r['COL']
    elif org == 'Salmonella Typhi':
        r['CRO'] = R(0.03); r['AMC'] = R(0.2); r['TZP'] = R(0.05); r['CSL'] = R(0.05)
        for d in ('ETP', 'IPM', 'MEM'): r[d] = 'S'
    elif org == 'Haemophilus influenzae':
        r['AMC'] = R(0.10); r['CRO'] = R(0.02); r['TZP'] = R(0.03); r['CSL'] = R(0.03)
        for d in ('ETP', 'IPM', 'MEM'): r[d] = 'S'
    elif org == 'Staphylococcus aureus':
        mrsa = rng.rand() < 0.45 * k; r['OXA'] = 'R' if mrsa else 'S'
        for d in ('AMC', 'CRO', 'TZP', 'CSL', 'ETP', 'IPM', 'MEM'): r[d] = 'R' if mrsa else 'S'
        r['VAN'] = 'S'; r['TEC'] = R(0.01); r['LNZ'] = R(0.01)
    elif org == 'Streptococcus pneumoniae':
        r['CRO'] = R(0.04); r['AMC'] = R(0.05); r['TZP'] = r['AMC']; r['CSL'] = R(0.05)
        for d in ('ETP', 'IPM', 'MEM'): r[d] = R(0.02)
        r['VAN'] = 'S'; r['TEC'] = 'S'; r['LNZ'] = 'S'
    elif org == 'Enterococcus spp.':
        amp = R(0.60); r['AMP'] = amp; r['AMC'] = amp; r['TZP'] = amp; r['IPM'] = amp
        for d in ('CRO', 'ETP', 'MEM', 'CSL'): r[d] = 'R'      # unreliable / intrinsic for enterococci
        r['VAN'] = R(0.08); r['TEC'] = r['VAN']; r['LNZ'] = R(0.01)
    return r

MIX = {  # organism counts per specimen type (ICU A); ICU B uses a scaled-down, less MDR mix
 'Blood': {'Escherichia coli': 80, 'Klebsiella pneumoniae': 90, 'Acinetobacter baumannii': 50, 'Pseudomonas aeruginosa': 35,
           'Staphylococcus aureus': 55, 'Enterococcus spp.': 30, 'Salmonella Typhi': 25},
 'Resp-CAP': {'Streptococcus pneumoniae': 40, 'Haemophilus influenzae': 25, 'Klebsiella pneumoniae': 25,
              'Staphylococcus aureus': 20, 'Pseudomonas aeruginosa': 10},
 'Resp-VAP': {'Klebsiella pneumoniae': 90, 'Acinetobacter baumannii': 110, 'Pseudomonas aeruginosa': 60,
              'Staphylococcus aureus': 25, 'Escherichia coli': 20},
 'Peritoneal': {'Escherichia coli': 70, 'Klebsiella pneumoniae': 45, 'Enterococcus spp.': 25,
                'Pseudomonas aeruginosa': 15, 'Acinetobacter baumannii': 10},
}

def make_icu(name, k, scale, seed, acin_factor=1.0):
    rng = np.random.RandomState(seed); rows = []; pid = 0
    for spec, mix in MIX.items():
        for org, n in mix.items():
            n = int(round(n * scale * (acin_factor if org == 'Acinetobacter baumannii' else 1)))
            for _ in range(n):
                pid += 1; base = dict(ICU=name, PATIENT_ID=f'{name}-{pid:05d}', SPEC_TYPE=spec, ORGANISM=org,
                                      SPEC_DATE=pd.Timestamp('2025-01-01') + pd.Timedelta(days=int(rng.randint(0, 365))))
                ph = phenotype(org, rng, k); rows.append({**base, **ph})
                if rng.rand() < 0.12:   # repeat isolate from same patient -> must be removed by first-isolate rule
                    rows.append({**base, 'SPEC_DATE': base['SPEC_DATE'] + pd.Timedelta(days=3), **phenotype(org, rng, k)})
    df = pd.DataFrame(rows)
    df.to_csv(f'{OUT}/{name}_whonet.csv', index=False)
    return df

if __name__ == '__main__':
    a = make_icu('icu_A', k=1.0, scale=1.0, seed=1)
    b = make_icu('icu_B', k=0.35, scale=0.7, seed=2, acin_factor=0.4)
    print('ICU A isolates', len(a), '| ICU B isolates', len(b), '| files in', OUT)

In [ ]:
!python make_data.py
regimens = pd.read_csv('data/icmr_regimens.csv'); policy = pd.read_csv('data/policy.csv')
SPEC_MAP = dict(pd.read_csv('data/specimen_map.csv').values)
ICUS = {'icu_A': pd.read_csv('data/icu_A_whonet.csv'), 'icu_B': pd.read_csv('data/icu_B_whonet.csv')}
display(regimens.head(8)); display(policy); display(ICUS['icu_A'].head())

## 3. The engine (deterministic core)

In [ ]:
%%writefile engine.py
"""
Antibiotic Policy Engine: deterministic, auditable core.

Merges three sources into one empiric recommendation per clinical syndrome:
  1. ICMR syndrome-wise regimens        (data/icmr_regimens.csv)
  2. The ICU's own cumulative antibiogram (data/<icu>_whonet_isolates.csv -> computed here)
  3. The hospital's stewardship policy    (data/policy.csv)

No machine learning makes the recommendation. Every number can be traced back
to a resistance percentage or a policy row, and the same input always gives
the same output.
"""
import json, hashlib
import pandas as pd
import numpy as np

SAFE_MAX, CAUTION_MAX = 10.0, 30.0          # % resistance thresholds for the three tiers
MIN_ISOLATES = 30                           # CLSI M39: fewer isolates -> estimate unreliable
TIER_NAMES = {1: 'SAFE', 2: 'USE WITH CAUTION', 3: 'NO RELIABLE EMPIRIC OPTION'}
AWARE_RANK = {'Access': 0, 'Watch': 1, 'Reserve': 2}
RESTRICT_RANK = {'free': 0, 'justify': 1, 'id_approval': 2}


# ---------------------------------------------------------------- antibiogram
def cumulative_antibiogram(isolates: pd.DataFrame) -> pd.DataFrame:
    """WHONET isolate export -> cumulative antibiogram (% resistant), CLSI M39 style.
    Keeps only the first isolate per patient per organism per specimen type."""
    first = isolates.sort_values('SPEC_DATE').drop_duplicates(['PATIENT_ID', 'ORGANISM', 'SPEC_TYPE'])
    abx_cols = [c for c in isolates.columns if c.isupper() and len(c) == 3 and c not in ('ICU',)]
    rows = []
    for (spec, org), g in first.groupby(['SPEC_TYPE', 'ORGANISM']):
        for a in abx_cols:
            v = g[a].dropna()
            if len(v) == 0:
                continue
            rows.append(dict(SPEC_TYPE=spec, ORGANISM=org, ANTIBIOTIC=a, N=len(v),
                             PCT_R=round(100 * (v != 'S').mean(), 1)))   # I counted as non-susceptible
    return pd.DataFrame(rows)


def organism_weights(isolates: pd.DataFrame) -> pd.DataFrame:
    """Share of each organism within a specimen type (used to weight resistance)."""
    first = isolates.drop_duplicates(['PATIENT_ID', 'ORGANISM', 'SPEC_TYPE'])
    w = first.groupby(['SPEC_TYPE', 'ORGANISM']).size().rename('N').reset_index()
    w['WEIGHT'] = w['N'] / w.groupby('SPEC_TYPE')['N'].transform('sum')
    return w


def regimen_resistance(isolates, spec, drugs):
    """Weighted-incidence syndromic resistance (WISCA idea), computed at isolate level:
    an isolate counts as covered if it is susceptible to AT LEAST ONE drug in the regimen.
    Untested drug/organism pairs (intrinsic resistance) count as not covered."""
    first = isolates.drop_duplicates(['PATIENT_ID', 'ORGANISM', 'SPEC_TYPE'])
    g = first[first.SPEC_TYPE == spec]
    if len(g) == 0:
        return None, 0, {}
    cov = np.zeros(len(g), bool)
    for d in drugs:
        if d in g.columns:
            cov |= (g[d] == 'S').values
    per_org = {}
    for org, gg in g.assign(_cov=cov).groupby('ORGANISM'):
        per_org[org] = round(100 * (1 - gg._cov.mean()), 1)
    return round(100 * (1 - cov.mean()), 1), len(g), per_org


def tier_of(pct_r):
    if pct_r is None: return 3
    return 1 if pct_r <= SAFE_MAX else (2 if pct_r <= CAUTION_MAX else 3)


# ---------------------------------------------------------------- engine
def recommend(syndrome, isolates, regimens, policy, specimen_map,
              exclude=(), patient_flags=()):
    """Return a ranked, fully explained recommendation for one syndrome in one ICU."""
    spec = specimen_map[syndrome]
    pol = policy.set_index('CODE')
    cands = regimens[regimens.SYNDROME == syndrome]
    out = []
    for _, r in cands.iterrows():
        drugs = r.DRUGS.split('+')
        reasons = []
        # patient-level exclusions (e.g. polymyxins in acute kidney injury, penicillin anaphylaxis)
        blocked = [d for d in drugs if d in exclude or any(f in pol.loc[d, 'CONTRAINDICATED_IN'].split('|') for f in patient_flags)]
        if blocked:
            out.append(dict(regimen=r.REGIMEN, drugs=drugs, line=r.LINE, status='EXCLUDED',
                            reason=f"contraindicated for this patient: {', '.join(pol.loc[b, 'NAME'] for b in blocked)}"))
            continue
        pct, n, per_org = regimen_resistance(isolates, spec, [d for d in drugs if pol.loc[d, 'CULTURE_EVALUABLE'] == 1])
        t = tier_of(pct)
        if n < MIN_ISOLATES:
            reasons.append(f'only {n} isolates (<{MIN_ISOLATES}): local estimate unreliable'); t = max(t, 2)
        aware = max((pol.loc[d, 'AWARE'] for d in drugs), key=lambda a: AWARE_RANK[a])
        restr = max((pol.loc[d, 'RESTRICTION'] for d in drugs), key=lambda a: RESTRICT_RANK[a])
        avail = all(pol.loc[d, 'FORMULARY'] == 1 for d in drugs)
        if not avail:
            out.append(dict(regimen=r.REGIMEN, drugs=drugs, line=r.LINE, status='EXCLUDED',
                            reason='not on hospital formulary')); continue
        if restr == 'id_approval': reasons.append('requires Infectious Disease approval')
        elif restr == 'justify': reasons.append('prescriber justification required')
        out.append(dict(regimen=r.REGIMEN, drugs=drugs, line=r.LINE, status='CANDIDATE',
                        pct_resistant=pct, isolates=n, per_organism=per_org, tier=t, tier_name=TIER_NAMES[t],
                        aware=aware, restriction=restr, reason='; '.join(reasons), dosing=r.DOSING))
    ok = [c for c in out if c['status'] == 'CANDIDATE']
    # stewardship ranking: safest tier, then narrowest spectrum (AWaRe), least restricted, fewest drugs, best coverage
    ok.sort(key=lambda c: (c['tier'], AWARE_RANK[c['aware']], RESTRICT_RANK[c['restriction']], len(c['drugs']), c['pct_resistant']))
    best = ok[0] if ok else None
    if best is None or best['tier'] == 3:
        decision = dict(action='ESCALATE',
                        message='No regimen in the guideline is reliably active in this ICU. Do not start a '
                                'potentially inactive empiric regimen: send cultures and call Microbiology / '
                                'Infectious Disease now.')
    else:
        decision = dict(action='RECOMMEND', regimen=best['regimen'], tier=best['tier_name'],
                        message=f"{best['regimen']}: {best['pct_resistant']}% of local {spec.lower()} isolates resistant"
                                + (f" ({best['reason']})" if best['reason'] else ''))
    result = dict(syndrome=syndrome, specimen=spec, decision=decision, ranked=ok,
                  excluded=[c for c in out if c['status'] == 'EXCLUDED'],
                  patient_flags=list(patient_flags))
    result['audit_hash'] = hashlib.sha256(json.dumps(result, sort_keys=True, default=str).encode()).hexdigest()[:16]
    return result


def pretty(res):
    lines = [f"\n=== {res['syndrome']}  (specimen: {res['specimen']}) ===",
             f"DECISION: {res['decision']['action']} | {res['decision']['message']}"]
    for i, c in enumerate(res['ranked'], 1):
        lines.append(f"  {i}. {c['regimen']:<45} {c['pct_resistant']:>5}% R  [{c['tier_name']}]  {c['aware']}"
                     + (f"  - {c['reason']}" if c['reason'] else ''))
    for c in res['excluded']:
        lines.append(f"  x  {c['regimen']:<45} EXCLUDED - {c['reason']}")
    lines.append(f"  audit hash: {res['audit_hash']}")
    return '\n'.join(lines)

In [ ]:
import importlib, engine as E; importlib.reload(E)
# Cumulative antibiograms (CLSI M39: first isolate per patient)
ab = {k: E.cumulative_antibiogram(v) for k, v in ICUS.items()}
for k, v in ICUS.items():
    print(f"{k}: {len(v)} raw isolates -> {len(v.drop_duplicates(['PATIENT_ID','ORGANISM','SPEC_TYPE']))} after first-isolate rule")
ab['icu_A'].head(10)

## 4. Cumulative antibiogram heatmaps (ICU A vs ICU B)

In [ ]:
SHOW = ['AMC','CRO','TZP','CSL','ETP','MEM','AMK','COL','VAN','LNZ']
fig, axes = plt.subplots(1, 2, figsize=(20, 7)); fig.subplots_adjust(wspace=0.6)
for ax, (k, a) in zip(axes, ab.items()):
    t = a[a.SPEC_TYPE.isin(['Blood','Resp-VAP'])].copy()
    t['ROW'] = t.SPEC_TYPE + ' | ' + t.ORGANISM
    p = t.pivot_table(index='ROW', columns='ANTIBIOTIC', values='PCT_R').reindex(columns=SHOW)
    im = ax.imshow(p.values, cmap='RdYlGn_r', vmin=0, vmax=100, aspect='auto')
    ax.set_xticks(range(len(SHOW))); ax.set_xticklabels(SHOW); ax.set_yticks(range(len(p))); ax.set_yticklabels(p.index, fontsize=8)
    for i in range(p.shape[0]):
        for j in range(p.shape[1]):
            v = p.values[i, j]
            if not np.isnan(v): ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=7)
    ax.set_title(f'{k}: % resistant')
plt.colorbar(im, ax=axes, shrink=0.7); plt.savefig('outputs/fig_antibiograms.png', dpi=150, bbox_inches='tight'); plt.show()

## 5. Recommendations for every syndrome in both ICUs

In [ ]:
results, rows = {}, []
for icu, iso in ICUS.items():
    for s in SPEC_MAP:
        r = E.recommend(s, iso, regimens, policy, SPEC_MAP); results[(icu, s)] = r
        best = r['ranked'][0] if r['ranked'] else {}
        rows.append(dict(ICU=icu, Syndrome=s, Decision=r['decision']['action'],
                         Regimen=r['decision'].get('regimen', 'Escalate to ID / Microbiology'),
                         Tier=r['decision'].get('tier', '-'), PctResistant=best.get('pct_resistant') if r['decision']['action']=='RECOMMEND' else None,
                         Audit=r['audit_hash']))
summary = pd.DataFrame(rows); summary.to_csv('outputs/recommendations_summary.csv', index=False)
display(summary)
print(E.pretty(results[('icu_A', 'Hospital-acquired / ventilator-associated pneumonia')]))
print(E.pretty(results[('icu_B', 'Hospital-acquired / ventilator-associated pneumonia')]))

### Patient-level overlay: same patient, but with acute kidney injury

In [ ]:
aki = E.recommend('Sepsis / septic shock (unclear focus)', ICUS['icu_A'], regimens, policy, SPEC_MAP, patient_flags=['aki'])
print(E.pretty(aki))

## 6. Validation tests
These are the checks behind the claim *"the merge logic correctly classifies tiers, applies policy and flags escalation"*.

In [ ]:
tests = []
def check(name, cond): tests.append((name, 'PASS' if cond else 'FAIL'))
check('Tier boundary: 10.0% -> SAFE', E.tier_of(10.0) == 1)
check('Tier boundary: 10.1% -> CAUTION', E.tier_of(10.1) == 2)
check('Tier boundary: 30.0% -> CAUTION', E.tier_of(30.0) == 2)
check('Tier boundary: 30.1% -> NO RELIABLE OPTION', E.tier_of(30.1) == 3)
a = ICUS['icu_A']
check('First-isolate rule removes repeat isolates', len(a.drop_duplicates(['PATIENT_ID','ORGANISM','SPEC_TYPE'])) < len(a))
r = results[('icu_A', 'Hospital-acquired / ventilator-associated pneumonia')]
check('Restricted drug carries ID-approval flag', 'Infectious Disease approval' in r['decision']['message'])
check('Escalates when every regimen is >30% resistant', all(c['tier'] == 3 for c in results[('icu_A','Intra-abdominal infection (secondary peritonitis)')]['ranked']) and results[('icu_A','Intra-abdominal infection (secondary peritonitis)')]['decision']['action'] == 'ESCALATE')
check('AKI flag removes colistin / polymyxin / amikacin', all(not set(c['drugs']) & {'COL','PMB','AMK'} for c in aki['ranked']))
check('Deterministic: same input -> same audit hash', E.recommend('Sepsis / septic shock (unclear focus)', a, regimens, policy, SPEC_MAP)['audit_hash'] == results[('icu_A','Sepsis / septic shock (unclear focus)')]['audit_hash'])
diff = sum(results[('icu_A', s)]['decision'].get('regimen') != results[('icu_B', s)]['decision'].get('regimen') for s in SPEC_MAP)
check('Different ICU data -> different recommendations (generalises)', diff >= 2)
cand = results[('icu_B','Sepsis / septic shock (unclear focus)')]
check('Stewardship ranking prefers Watch over Reserve when both are SAFE', cand['ranked'][0]['aware'] != 'Reserve')
tdf = pd.DataFrame(tests, columns=['Test', 'Result']); tdf.to_csv('outputs/validation_tests.csv', index=False); display(tdf)
print(f"{(tdf.Result=='PASS').sum()}/{len(tdf)} tests passed")

## 7. RAG over the ICMR guideline (LangChain + Ollama + FAISS)
Used only to **explain**, never to decide. It downloads the ICMR PDF, chunks it, embeds the chunks with `nomic-embed-text`, stores them in FAISS, and answers with a small local LLM (`qwen2.5:1.5b`) that must cite page numbers.
If Ollama or the PDF download fails, it falls back to TF-IDF retrieval (no LLM) so the notebook still finishes.

In [ ]:
ICMR_URL = 'https://www.icmr.gov.in/icmrobject/custom_data/pdf/resource-guidelines/Treatment_Guidelines_2019_Final.pdf'
pdf_ok = False
try:
    req = urllib.request.Request(ICMR_URL, headers={'User-Agent': 'Mozilla/5.0'})
    blob = urllib.request.urlopen(req, timeout=120).read()
    if blob[:4] == b'%PDF': open('icmr_2019.pdf', 'wb').write(blob)
except Exception as e:
    print('ICMR PDF download failed:', e, '-> you can upload it manually as icmr_2019.pdf and re-run this cell')
pdf_ok = os.path.exists('icmr_2019.pdf') and os.path.getsize('icmr_2019.pdf') > 10000
from pypdf import PdfReader
pages = [(i + 1, p.extract_text() or '') for i, p in enumerate(PdfReader('icmr_2019.pdf').pages)] if pdf_ok else []
if not pages:   # small built-in summary of stewardship principles so the RAG demo still runs
    BUILTIN = [
     'Collect blood and site cultures before the first antibiotic dose whenever this does not delay treatment of septic shock.',
     'Review empiric therapy at 48 to 72 hours. De-escalate to the narrowest active agent once culture and susceptibility results are known, and stop antibiotics if infection is ruled out.',
     'Reserve agents such as colistin and polymyxin B should be used only for confirmed or strongly suspected carbapenem-resistant Gram-negative infection, ideally after infectious disease or microbiology input, and with renal monitoring.',
     'Empiric choices should be guided by the local cumulative antibiogram of the unit, because national guidelines cannot reflect resistance in a particular ICU.',
     'Septic shock warrants prompt empiric combination therapy with at least two antibiotics of different classes, followed by early de-escalation.',
     'Use the shortest effective duration; most uncomplicated sepsis, pneumonia and intra-abdominal infections with source control need 5 to 8 days.']
    pages = [(0, t) for t in BUILTIN]; print('Using built-in stewardship summary (page 0 = not from the PDF)')
chunks = []
for pg, txt in pages:
    for k in range(0, len(txt), 1000):
        chunks.append((pg, txt[k:k + 1200]))
print(len(pages), 'pages ->', len(chunks), 'chunks')

In [ ]:
# Try to start Ollama (local LLM server). Takes ~3-5 min the first time.
USE_LLM = False
try:
    os.system('apt-get -qq install -y zstd pciutils > /dev/null 2>&1')
    os.system('curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1')
    import subprocess; subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL); time.sleep(8)
    assert os.system('ollama pull nomic-embed-text > /dev/null 2>&1') == 0
    assert os.system('ollama pull qwen2.5:1.5b > /dev/null 2>&1') == 0
    from langchain_ollama import OllamaEmbeddings, ChatOllama
    from langchain_community.vectorstores import FAISS
    store = FAISS.from_texts([c[1] for c in chunks], OllamaEmbeddings(model='nomic-embed-text'), metadatas=[{'page': c[0]} for c in chunks])
    llm = ChatOllama(model='qwen2.5:1.5b', temperature=0)
    USE_LLM = True; print('Ollama + FAISS ready')
except Exception as e:
    print('Ollama unavailable -> using TF-IDF retrieval only.', e)
from sklearn.feature_extraction.text import TfidfVectorizer
tfv = TfidfVectorizer(stop_words='english').fit([c[1] for c in chunks]) if chunks else None
tfm = tfv.transform([c[1] for c in chunks]) if chunks else None

def retrieve(q, k=4):
    if USE_LLM:
        return [(d.metadata['page'], d.page_content) for d in store.similarity_search(q, k=k)]
    if tfv is None: return []
    s = (tfm @ tfv.transform([q]).T).toarray().ravel()
    return [chunks[i] for i in s.argsort()[::-1][:k]]

def ask(q):
    ctx = retrieve(q)
    if not ctx: return 'No guideline text available.'
    if not USE_LLM:
        return 'Top passages (no LLM):\n' + '\n---\n'.join(f'[p.{p}] {t[:400]}...' for p, t in ctx)
    context = '\n\n'.join(f'[page {p}] {t}' for p, t in ctx)
    prompt = (f'You are an antimicrobial stewardship assistant. Answer ONLY from the ICMR guideline context below. '
              f'Cite page numbers like (p. 12). If the answer is not in the context, say so. Never invent doses.\n\n'
              f'CONTEXT:\n{context}\n\nQUESTION: {q}')
    return llm.invoke(prompt).content

def explain(result):
    d = result['decision']
    q = (f"Why might an ICU escalate to an infectious disease consult instead of starting empiric therapy for {result['syndrome']}?"
         if d['action'] == 'ESCALATE' else f"What does ICMR say about using {d['regimen']} for {result['syndrome']}?")
    return ask(q)

for q in ['Why is de-escalation important after culture results?', 'When should colistin be used in sepsis?']:
    print('Q:', q); print(ask(q)); print('=' * 80)
print('EXPLANATION for ICU A VAP decision:'); print(explain(results[('icu_A', 'Hospital-acquired / ventilator-associated pneumonia')]))

## 8. Optional: interactive demo (public Gradio link, valid 72 h)

In [ ]:
import gradio as gr
def ui(icu, syndrome, aki, pen):
    flags = (['aki'] if aki else []) + (['penicillin_anaphylaxis'] if pen else [])
    r = E.recommend(syndrome, ICUS[icu], regimens, policy, SPEC_MAP, patient_flags=flags)
    return '```\n' + E.pretty(r) + '\n```'
gr.Interface(ui, [gr.Dropdown(list(ICUS), value='icu_A', label='ICU'), gr.Dropdown(list(SPEC_MAP), value=list(SPEC_MAP)[1], label='Syndrome'),
                  gr.Checkbox(label='Acute kidney injury'), gr.Checkbox(label='Penicillin anaphylaxis')],
             gr.Markdown(), title='Antibiotic Policy Engine').launch(share=True, debug=False)

## 9. Export

In [ ]:
json.dump({f'{k[0]} | {k[1]}': v for k, v in results.items()}, open('outputs/recommendations_full.json', 'w'), indent=2, default=str)
for k, a in ab.items(): a.to_csv(f'outputs/antibiogram_{k}.csv', index=False)
with zipfile.ZipFile('antibiotic_engine_outputs.zip', 'w') as z:
    for f in os.listdir('outputs'): z.write(f'outputs/{f}', f)
    for f in os.listdir('data'): z.write(f'data/{f}', f'data/{f}')
    z.write('engine.py'); z.write('make_data.py')
try:
    from google.colab import files; files.download('antibiotic_engine_outputs.zip')
except Exception: pass
print('done')